In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e4/sample_submission.csv
/kaggle/input/playground-series-s5e4/train.csv
/kaggle/input/playground-series-s5e4/test.csv


In [16]:
train = pd.read_csv("../input/playground-series-s5e4/train.csv")
test = pd.read_csv("../input/playground-series-s5e4/test.csv")

In [17]:
# Define preprocessing function to apply the same steps to both train and test datasets
def preprocess(df, medians=None, is_train=True):
    df = df.copy()

    # Fill missing values with precomputed or current medians
    if medians is None and is_train:
        medians = {
            'Episode_Length_minutes': df['Episode_Length_minutes'].median(),
            'Guest_Popularity_percentage': df['Guest_Popularity_percentage'].median(),
            'Number_of_Ads': df['Number_of_Ads'].median()
        }

    df['Episode_Length_minutes'] = df['Episode_Length_minutes'].fillna(medians['Episode_Length_minutes'])
    df['Guest_Popularity_percentage'] = df['Guest_Popularity_percentage'].fillna(medians['Guest_Popularity_percentage'])
    df['Number_of_Ads'] = df['Number_of_Ads'].fillna(medians['Number_of_Ads'])

    # Feature engineering: calculate popularity difference
    df['popularity_diff'] = df['Host_Popularity_percentage'] - df['Guest_Popularity_percentage']

    # Create time period category from publication time
    def get_time_period(time_str):
        try:
            if pd.isnull(time_str):
                return "unknown"
            h = int(time_str.split(":")[0])
            return "morning" if h < 12 else "afternoon" if h < 18 else "evening"
        except:
            return "unknown"

    df['Time_Period'] = df['Publication_Time'].apply(get_time_period)

    return df, medians

In [18]:
# Apply preprocessing to train and test
train, medians = preprocess(train, is_train=True)
test, _ = preprocess(test, medians, is_train=False)

In [19]:
# Separate target variable and drop unused columns
y = train['Listening_Time_minutes']
X = train.drop(columns=['Listening_Time_minutes', 'id'])
test_ids = test['id']
test = test.drop(columns=['id'])

In [20]:
# Concatenate train and test for consistent encoding
combined = pd.concat([X, test], axis=0)
combined = pd.get_dummies(combined)

In [21]:
# Split back to train and test
X = combined[:len(X)]
X_test = combined[len(X):]

In [22]:
# Split training set into training and validation subsets
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
# Define LightGBM model
model = LGBMRegressor(
    random_state=42,
    n_estimators=500,        # number of trees
    learning_rate=0.05,      # learning rate (smaller = more accurate)
    max_depth=6,             # control model complexity
    subsample=0.8,           # use 80% of samples for each tree
    colsample_bytree=0.8     # use 80% of features for each tree
)

In [24]:
# Train the model
model.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.037008 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1369
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 177
[LightGBM] [Info] Start training from score 45.447808


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, max_depth=6,
              n_estimators=500, random_state=42, subsample=0.8)

In [25]:
# Evaluate on validation set
y_pred = model.predict(X_valid)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_valid, y_pred)

In [26]:
# Output validation results
print("Validation Results:")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

Validation Results:
MSE: 169.98
RMSE: 13.04
R2 Score: 0.7690


In [29]:
# Generate predictions on test data
final_preds = model.predict(X_test)

In [30]:
# Create submission DataFrame
submission = pd.DataFrame({
    'id': test_ids,
    'Listening_Time_minutes': final_preds
})

In [31]:
# Save submission file
submission.to_csv("submission.csv", index=False)